In [45]:
import sys
sys.path.append("../")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.colors import LightSource
from sklearn.preprocessing import QuantileTransformer
from sklearn.cluster import KMeans
import skgstat as skg
from skgstat import models
import gstatsim_torch as gst
import parallel_torch as gspt
import torch
import math
import time

In [46]:
from torch.profiler import profile, record_function, ProfilerActivity

In [47]:
import random

## Data Preparation

In [48]:
df_bed = pd.read_csv('demos/data/greenland_test_data.csv')

# remove erroneously high values due to bad bed picks
df_bed = df_bed[df_bed["Bed"] <= 700]  

In [49]:
# grid data to 100 m resolution and remove coordinates with NaNs
res = 1000
df_grid, torch_data, rows, cols = gst.Gridding.grid_data(df_bed, 'X', 'Y', 'Bed', res)
df_grid = df_grid[df_grid["Z"].isnull() == False]
torch_data = torch_data[torch_data[:,2].isnan() == False]
df_grid = df_grid.rename(columns = {"Z": "Bed"})

# normal score transformation
data = df_grid['Bed'].values.reshape(-1,1)
nst_trans = QuantileTransformer(n_quantiles=500, output_distribution="normal").fit(data)
df_grid['Nbed'] = nst_trans.transform(data)

In [50]:
# define coordinate grid
xmin = torch.min(torch_data[:,0]); xmax = torch.max(torch_data[:,0])     # min and max x values
ymin = torch.min(torch_data[:,1]); ymax = torch.max(torch_data[:,1])     # min and max y values

Pred_grid_xy = gst.Gridding.prediction_grid(xmin, xmax, ymin, ymax, res)

print(xmin, ymin, Pred_grid_xy.shape)

tensor(-300000.) tensor(-1800000.) torch.Size([22500, 2])


In [51]:
# K means clustering
n_clusters = 3
kmeans = KMeans(n_clusters = n_clusters, random_state = 0, n_init = 10).fit(df_grid[['X','Y','Nbed']])
df_grid['K'] = kmeans.labels_  # make column in dataframe with cluster name

# add partition labels to the last column
torch_data = torch.column_stack((torch_data, torch.from_numpy(kmeans.labels_)))

# experimental variogram parameters
maxlag = 50000
n_lags = 70 #num of bins

# cluster 0 variogram
df0 = df_grid[df_grid['K'] == 0]
coords0 = df0[['X','Y']].values
values0 = df0['Nbed']
V0 = skg.Variogram(coords0, values0, bin_func = "even", n_lags = n_lags, 
                   maxlag = maxlag, normalize=False)


# cluster 1 variogram
df1 = df_grid[df_grid['K'] == 1]
coords1 = df1[['X','Y']].values
values1 = df1['Nbed']
V1 = skg.Variogram(coords1, values1, bin_func = "even", n_lags = n_lags, 
                   maxlag = maxlag, normalize=False)


# cluster 2 variogram
df2 = df_grid[df_grid['K'] == 2]
coords2 = df2[['X','Y']].values
values2 = df2['Nbed']
V2 = skg.Variogram(coords2, values2, bin_func = "even", n_lag = n_lags, 
                   maxlag = maxlag, normalize=False) 

range0 = V0.parameters[0]; sill0 = V0.parameters[1]
range1 = V1.parameters[0]; sill1 = V1.parameters[1]
range2 = V2.parameters[0]; sill2 = V2.parameters[1]

# make a list with variogram parameters
azimuth = 0

# nugget effect
nug = 0 

# variogram model
vtype = 'Exponential'

# define variograms for each cluster
# Azimuth, nugget, major range, minor range, sill
gam0 = [azimuth, nug, range0, range0, sill0, vtype]
gam1 = [azimuth, nug, range1, range1, sill1, vtype]
gam2 = [azimuth, nug, range2, range2, sill2, vtype]

# store variogram parameters
#df_gamma = pd.DataFrame({'Variogram': [gam0, gam1, gam2]})
gamma = torch.tensor([gam0[:5], gam1[:5], gam2[:5]]).cuda()

In [52]:
k = 48         # number of neighboring data points used to estimate a given point
rad = 50000     # 50 km search radius

## Vectorized and Parallel torch scheme
Not actually parallelized with starmap but set up so that it can easily be converted

In [53]:
# convert parameters to simulation function naming scheme
prediction_grid = Pred_grid_xy
torch_data = torch_data
num_points = k
gamma = gamma
radius = rad
num_gpus = 3 * 4# more than actual to not run into memory constraints (make batch tensor smaller)

In [54]:
# setup before parallel function

# input torch_data, prediction_grid

observed_coords = torch_data[:,:2].tolist() # xy
simulate_coords = [coord for coord in prediction_grid.tolist() if coord not in observed_coords] # xy

observed_coords = torch.tensor(observed_coords)
simulate_coords = torch.tensor(simulate_coords)

print(f"observed_coords: {observed_coords.shape}")
print(f"simulate coords: {simulate_coords.shape}")

# Shuffle data to predict to create a random path
index = torch.arange(len(simulate_coords)) 
shuffle = index[torch.randperm(len(simulate_coords))]
simulate_coords = simulate_coords[shuffle]

full = torch.vstack((observed_coords, simulate_coords))

K_list = torch_data[:,3]

print(f"full = {full.shape}")

rotation_matrix = torch.zeros((len(gamma), 2, 2))

for i, vario in enumerate(gamma):

    azimuth = vario[0]
    major_range = vario[2]
    minor_range = vario[3]

    rotation_matrix[i] = gst.make_rotation_matrix(azimuth, major_range, minor_range, "cpu")

# create starting index for data from full to use for KNN
begin = len(observed_coords)

num_cells = len(simulate_coords)
cells_per_process = num_cells//num_gpus  # 
#print(f"cells_per_process={cells_per_process}") # 18850/12 = 1570 per GPU.

i_list = [[i for i in range(j*cells_per_process, (j+1)*cells_per_process)] for j in range(num_gpus-1)]
i_list.append([i for i in range((num_gpus-1)*cells_per_process,num_cells)])

# print(len(i_list), len(i_list[0])) # 12 GPU x 1570 cells/GPU
gpu_num = [i for i in range(num_gpus)] # [0,1,2,...11]


observed_coords: torch.Size([3650, 2])
simulate coords: torch.Size([18850, 2])
full = torch.Size([22500, 2])


In [55]:
# convert parameters to parallel function naming scheme
i_list = i_list[0] # [1,...,1570]
gpu_id = gpu_num[0] # 0
full = full
gamma = gamma
radius = radius
num_points = num_points
begin = begin
rotation_matrix = rotation_matrix

In [56]:
def preprocess_for_nn_search(search_candidates, loc, radius, num_points):
    """
    Preprocess input search_candidates and loc 
    input search_candidates, loc, radius, num_points
    return:
        stack
        indices,
        bins
        bin_indices
        oct_count
    """
    B, N, _ = search_candidates.shape

    # Repeat loc to align shapes for distance computation
    locx = loc[:, 0].unsqueeze(1).repeat(1, N)
    locy = loc[:, 1].unsqueeze(1).repeat(1, N)

    # Extract x/y coordinates
    x_tensor = search_candidates[:, :, 0]
    y_tensor = search_candidates[:, :, 1]

    # Compute distance and angle from loc
    centered_x = x_tensor - locx
    centered_y = y_tensor - locy
    distances = torch.sqrt(centered_x**2 + centered_y**2)
    angles = torch.atan2(centered_y, centered_x)

    # Stack into (B, N, 4): x, y, dist, angle
    stack = torch.stack((x_tensor, y_tensor, distances, angles), dim=2)

    # Create index tensor (B, N)
    indices = torch.arange(N, device=stack.device).unsqueeze(0).repeat(B, 1).float()

    # Mask out points beyond the radius
    mask = torch.where(distances < radius, 1.0, float('nan'))
    stack = stack * mask.unsqueeze(2).repeat(1, 1, 4)
    indices = indices * mask

    # Sort by distance
    sorted_dist_idxs = torch.argsort(stack[..., 2], dim=1)
    stack_idxs = torch.arange(B, device=stack.device).repeat_interleave(N).reshape(B, N)

    # Apply sorting
    stack = stack.gather(1, sorted_dist_idxs.unsqueeze(-1).expand(-1, -1, 4))
    indices = indices.gather(1, sorted_dist_idxs)

    # Define 8 bins over angle range
    bins = torch.tensor([
        -math.pi, -3*math.pi/4, -math.pi/2, -math.pi/4, 0,
         math.pi/4, math.pi/2,  3*math.pi/4, math.pi
    ], device=stack.device)

    # Bin index based on angle (angle at index 3)
    bin_indices = torch.bucketize(stack[..., 3], bins, right=False)

    # Octant point count
    oct_count = num_points // 8

    return stack, indices, bins, bin_indices, oct_count


In [57]:
# this is the vectorized version
def nn_search_vectorized(stack, indices, bin_indices, bins, num_points, oct_count):
    '''
    Vectorized nearest-neighbor search with binning.

    Parameters:
    - stack: (B, N, 3), where each point has [x, y, distance]
    - indices: (B, N), index of each point
    - bin_indices: (B, N), bin assignment for each point (1 to K)
    - bins: (K+1,), bin edges
    - num_points: total points to select per batch (K * oct_count)
    - oct_count: max number of closest neighbors to select from each bin

    Returns:
    - smallest: (B, num_points, 2), selected coordinates
    - index_list: (B, num_points), selected indices
    - vec_time: elapsed time for execution
    '''
    B, N, _ = stack.shape
    K = bins.shape[0] - 1  # number of bins

    start = time.time()

    # === Step 1: Create 3D bin mask ===
    # bin_mask[b, n, k] = 1 if point n in batch b is in bin k+1
    bin_mask = (bin_indices.unsqueeze(-1) == torch.arange(1, K+1, device=stack.device)).float()  # (B, N, K)
    
    # === Step 2: Apply mask to (x,y) and index values ===
    nan_mask = bin_mask.masked_fill(bin_mask == 0, float('nan'))  # Replace non-bin entries with NaN

    # Coordinates: (B, N, K, 2)
    masked_xy = stack[..., :2].unsqueeze(2) * nan_mask.unsqueeze(-1)
    
    # Indices: (B, N, K)
    masked_idx = indices.unsqueeze(2) * nan_mask

    # === Step 3: Count valid points per bin and clamp to oct_count ===
    is_valid = ~torch.isnan(masked_idx)
    bin_counts = is_valid.sum(dim=1)  # (B, K)
    clamped_counts = torch.clamp(bin_counts, max=oct_count)  # (B, K)

    # === Step 4: Sort distances inside bins ===
    masked_dist = stack[..., 2].unsqueeze(2) * nan_mask  # (B, N, K)
    sorted_dist, sorted_idx = torch.sort(masked_dist, dim=1)  # (B, N, K)
    topk_idx = sorted_idx[:, :oct_count, :]  # (B, oct_count, K)

    # === Step 5: Gather top-k points and indices ===
    b_idx = torch.arange(B, device=stack.device).view(B, 1, 1).expand(B, oct_count, K)
    k_idx = torch.arange(K, device=stack.device).view(1, 1, K).expand(B, oct_count, K)

    topk_xy = masked_xy[b_idx, topk_idx, k_idx, :]   # (B, oct_count, K, 2)
    topk_ids = masked_idx[b_idx, topk_idx, k_idx]    # (B, oct_count, K)

    # === Step 6: Mask out unused slots if bin has < oct_count points ===
    topk_range = torch.arange(oct_count, device=stack.device).view(1, -1, 1)
    valid_topk_mask = (topk_range < clamped_counts.unsqueeze(1)).float()  # (B, oct_count, K)

    topk_xy *= valid_topk_mask.unsqueeze(-1)
    topk_ids *= valid_topk_mask

    # === Step 7: Reshape results ===
    smallest = topk_xy.permute(0, 2, 1, 3).reshape(B, num_points, 2)
    index_list = topk_ids.permute(0, 2, 1).reshape(B, num_points)

    vec_time = time.time() - start
    return smallest, index_list, vec_time


In [58]:
def exponential_covariance(effective_lag, sill, nug):
    return (sill - nug)*torch.exp(-3 * effective_lag)

In [59]:
def make_covariance_matrix(smallest, vario, rotation_matrix):
    """
    Make covariance matrix showing covariances between each pair of input coordinates

    Parameters
    ----------
        smallest : (B, num_points, 2)
        vario : list of variogram parameters [azimuth, nugget, major_range, minor_range, sill, vtype]
        rotation_matrix : (2,2) matrix used to perform coordinate transformations

    Returns
    -------
        covariance_matrix : (B, num_points, num_points) matrix of covariance between n points
    """
    
    # Get column in batch variogram and expand dimensions to be compatible with effective_lag
    nug = vario[:,1].view(-1,1,1)
    sill = vario[:,4].view(-1,1,1)
    
    mat = torch.matmul(smallest, rotation_matrix)
    effective_lag = torch.cdist(mat, mat, p=2)  # Compute pairwise distances
    covariance_matrix = exponential_covariance(effective_lag, sill, nug)

    return covariance_matrix

In [60]:
def make_covariance_array(coord1, coord2, vario, rotation_matrix):
    """
    Make covariance array showing covariances between each data points and grid cell of interest

    Parameters
    ----------
        coord1 : numpy.ndarray
            coordinates of n data points
        coord2 : numpy.ndarray
            coordinates of grid cell of interest (i.e. grid cell being simulated) that is repeated n times
        vario : list
            list of variogram parameters [azimuth, nugget, major_range, minor_range, sill, vtype]
            azimuth, nugget, major_range, minor_range, and sill can be int or float type
            vtype is a string that can be either 'Exponential', 'Spherical', or 'Gaussian'
        rotation_matrix - rotation matrix used to perform coordinate transformations

    Returns
    -------
        covariance_array : numpy.ndarray
            nx1 array of covariance between n points and grid cell of interest
    """

    # Get column in batch variogram and expand dimensions to be compatible with effective_lag
    nug = vario[:,1].view(-1,1)
    sill = vario[:,4].view(-1,1)

    mat1 = torch.matmul(coord1, rotation_matrix)
    mat2 = torch.matmul(coord2, rotation_matrix)
    effective_lag = torch.sqrt(torch.sum((mat1 - mat2).pow(2), dim=2))
    covariance_array = exponential_covariance(effective_lag, sill, nug)

    return covariance_array

In [61]:
def reorder(covariance_matrix, covariance_array, index_vec, num_points):
    
    # Get mask to sort batch array and matrix so that nan are at the end
    nan_mask = torch.isnan(covariance_array)
    num_mask = ~torch.isnan(covariance_array)
    nan_indices = torch.nonzero(nan_mask)
    num_indices = torch.nonzero(num_mask)
    split_indices = torch.cat((num_indices, nan_indices))
    reorder_indices = split_indices[split_indices[:, 0].sort()[1]]
    reorder_indices = reorder_indices[:,1].reshape(covariance_array.shape)

    sorted_cov_array = covariance_array.gather(1, reorder_indices)
    sorted_index_vec = index_vec.gather(1, reorder_indices)

    reorder_rows = reorder_indices.unsqueeze(-1).repeat(1,1,num_points)
    reoreder_cols = reorder_indices.unsqueeze(1).repeat(1,num_points,1)

    sorted_cov_matrix = covariance_matrix.gather(1,reorder_rows).gather(2, reoreder_cols)
    
    return sorted_cov_matrix, sorted_cov_array, sorted_index_vec

In [62]:
def solve_system(sorted_cov_matrix, sorted_cov_array, num_points):
    
    # prepare for least squares by converting nan elements to identity

    lstsq_cov_array = sorted_cov_array.nan_to_num(1)

    identity_matrix = torch.eye(num_points, num_points).unsqueeze(0).repeat(len(sorted_cov_array),1,1).cuda()
    lstsq_cov_matrix = torch.where(torch.isnan(sorted_cov_matrix), identity_matrix, sorted_cov_matrix)
    
    k_weights = torch.linalg.lstsq(lstsq_cov_matrix, lstsq_cov_array).solution
    
    return k_weights

In [63]:
# Assign the GPU
torch.cuda.set_device(gpu_id)

# send data to GPU 
full = full.cuda()
rotation_matrix = rotation_matrix.cuda()

# Create batch tensors
offset = torch.tensor([i+begin for i in i_list]).cuda()
loc = full[offset]

# Get the size of the search matrix 
N = int(offset[-1])

search_candidates = torch.full((len(i_list), N, 2), float('nan')).cuda()
for i, cur_offset in enumerate(offset):
    search_candidates[i, :cur_offset] = full[:cur_offset]

In [64]:
# Given: search_candidates, loc, radius, num_points
stack, indicies, bins, bin_indices, oct_count = preprocess_for_nn_search(search_candidates, loc, radius, num_points)

# Run vectorized version
smallest_vec, index_vec, t_vec = nn_search_vectorized(stack, indicies, bin_indices, bins, num_points, oct_count)

In [65]:
# Get variogram cluster numbers from conditioning data
K_list = torch_data[:,3].cuda()

# Exclude indicies that are not original conditioning data
valid_cluster_indicies = torch.where(indicies < begin, indicies, float('nan'))

rand_K = torch.zeros(len(i_list), dtype=int).cuda()

# Select a random cluster from NN 
for i, row in enumerate(valid_cluster_indicies):
    row = row[~torch.isnan(row)]
    rand_index = row[random.randint(0,len(row)-1)]
    rand_K[i] = K_list[int(rand_index)]

In [66]:
# Create batch rotation matrix and variogram parameter list based on cluster number
batch_rot_mat = rotation_matrix[rand_K]
batch_gamma = gamma[rand_K]

In [67]:
# Get covariance matrix
covariance_matrix = make_covariance_matrix(smallest_vec, batch_gamma, batch_rot_mat)


# Covariance between data and unknown
covariance_array = make_covariance_array(smallest_vec, 
                loc.unsqueeze(1).repeat(1, num_points, 1), 
                batch_gamma, 
                batch_rot_mat
            )

In [68]:
# Reorder covariance matrix and array so NANs appear at the end
sorted_cov_matrix, sorted_cov_array, sorted_index_vec = reorder(covariance_matrix, covariance_array, index_vec, num_points)

# Get size list of number of NN collected for future indexing
size_list = torch.sum((~torch.isnan(sorted_cov_array)).int(), dim=1)

# Solve system defined by batch covariance matrix and array to get kriging weights
k_weights = solve_system(sorted_cov_matrix, sorted_cov_array, num_points)